# Auditoria e STL preliminar

Este notebook executa a leitura dos alvos estruturais e uma STL preliminar somente na Base 1, que já é diária e completa. As demais bases são listadas sem regularização ou imputação.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'processed').exists(), 'Execute a partir da raiz do repositório.'
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL

from discovery_series_temporais.series import load_candidate_target

pd.set_option('display.max_columns', None)
print(f'Raiz do projeto: {ROOT}')

Raiz do projeto: C:\Users\mayumishimizu-ieg\.codex\.chatgpt-projects\g-p-6ad36ceb09bc81918ea40186a7712f30\DiscoverySeriesTemporais


In [2]:
summaries = []
for base_id in range(1, 6):
    candidate = load_candidate_target(base_id, ROOT / 'data' / 'processed')
    summaries.append({
        'base': base_id,
        'target': candidate.target_name,
        'frequency_interpretation': candidate.frequency_label,
        'rows': len(candidate.data),
        'missing_target': int(candidate.data['target'].isna().sum()),
        'start': candidate.data['timestamp'].min(),
        'end': candidate.data['timestamp'].max(),
    })

summary = pd.DataFrame(summaries)
display(summary)

,base,target,frequency_interpretation,rows,missing_target,start,end
0,1,Close,daily,2836,0,2017-01-01 00:00:00,2024-10-06 00:00:00
1,2,traffic_volume,hourly (target-only consolidation),40575,0,2012-10-02 09:00:00,2018-09-30 23:00:00
2,3,PM2.5,hourly,35064,925,2013-03-01 00:00:00,2017-02-28 23:00:00
3,4,T (degC),hourly mean (exploratory),70129,88,2009-01-01 00:00:00,2017-01-01 00:00:00
4,5,GOLD_PRICE,next trading observation,9864,0,1968-04-29 00:00:00,2014-04-09 00:00:00


In [3]:
base_1 = load_candidate_target(1, ROOT / 'data' / 'processed')
series = base_1.data.set_index('timestamp')['target']
stl = STL(series, period=7, robust=True).fit()
seasonal_strength = max(0.0, 1.0 - np.var(stl.resid) / np.var(stl.resid + stl.seasonal))

figure = stl.plot()
figure.set_size_inches(12, 8)
figure.suptitle('Base 1 — STL preliminar com período semanal', y=1.02)
figure.tight_layout()
output = ROOT / 'reports' / 'generated' / 'figures' / 'base1_stl_weekly.png'
output.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(output, dpi=150, bbox_inches='tight')
plt.close(figure)

print(f'Força sazonal semanal preliminar: {seasonal_strength:.6f}')
print(f'Figura gerada: {output.relative_to(ROOT)}')
print('A STL das Bases 2 a 5 permanece pendente da decisão de agregação, lacunas ou calendário de negociação.')

Força sazonal semanal preliminar: 0.000000
Figura gerada: reports\generated\figures\base1_stl_weekly.png
A STL das Bases 2 a 5 permanece pendente da decisão de agregação, lacunas ou calendário de negociação.
